# A.1: Viterbi segmentation under a unigram model

Find the most probable segmentation with dynamic programming, then enumerate all segmentations of a short word to check it.


In [ ]:
import math

def viterbi(text, logp, max_len=10):
    """Most probable segmentation of `text` under a unigram model."""
    n = len(text)
    best = [0.0] + [-math.inf] * n     # best[j]: best log-prob of text[:j]
    back = [0] * (n + 1)
    for end in range(1, n + 1):
        for start in range(max(0, end - max_len), end):
            piece = text[start:end]
            if piece in logp and best[start] + logp[piece] > best[end]:
                best[end] = best[start] + logp[piece]
                back[end] = start
    pieces, end = [], n
    while end > 0:
        pieces.append(text[back[end]:end])
        end = back[end]
    return pieces[::-1], best[n]


In [ ]:
probs = {"h": .05, "u": .05, "g": .05, "s": .10, "hu": .05, "ug": .10,
         "gs": .05, "hug": .15, "ugs": .05, "b": .05, "un": .10, "bun": .20}
logp = {k: math.log(v) for k, v in probs.items()}
pieces, lp = viterbi("hugs", logp)
print(pieces, f"{math.exp(lp):.6f}")


**Expected output**

Output:

```text
['hug', 's'] 0.015000
```


In [ ]:
def all_segmentations(text, logp):
    if not text:
        yield [], 0.0
        return
    for i in range(1, len(text) + 1):
        if text[:i] in logp:
            for rest, lp in all_segmentations(text[i:], logp):
                yield [text[:i]] + rest, logp[text[:i]] + lp

segs = sorted(all_segmentations("hugs", logp), key=lambda s: -s[1])
Z = sum(math.exp(lp) for _, lp in segs)
for s, lp in segs:
    print(s, f"{math.exp(lp):.6f}", f"{math.exp(lp) / Z:.3f}")


**Expected output**

Output:

```text
['hug', 's'] 0.015000 0.718
['h', 'ugs'] 0.002500 0.120
['hu', 'gs'] 0.002500 0.120
['h', 'ug', 's'] 0.000500 0.024
['hu', 'g', 's'] 0.000250 0.012
['h', 'u', 'gs'] 0.000125 0.006
['h', 'u', 'g', 's'] 0.000013 0.001
```
